In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
import json

In [2]:
spark = SparkSession.builder \
    .appName("Telecom_Usage_Billing_Analytics") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

In [3]:
customers_data = """customer_id,customer_name,city,state,age,gender,plan_id,status
101,Rahul Sharma,Hyderabad,Telangana,35,Male,P101,Active
102,Priya Reddy,Bangalore,Karnataka,29,Female,P102,Active
103,Amit Kumar,Mumbai,Maharashtra,42,Male,P103,Inactive
104,Sneha Patel,Chennai,Tamil Nadu,31,Female,P101,Active
105,Farhan Ali,Delhi,Delhi,55,Male,P104,Active
106,Neha Singh,Pune,Maharashtra,38,Female,P102,Active
107,Arjun Verma,Hyderabad,Telangana,26,Male,P103,Inactive
108,Meera Nair,Kochi,Kerala,48,Female,P104,Active
109,Kiran Rao,Bangalore,Karnataka,33,Male,P101,Active
110,Nisha Reddy,Delhi,Delhi,41,Female,P102,Active
111,Ravi Kumar,Mumbai,Maharashtra,45,Male,P105,Active
112,Ayesha Khan,Hyderabad,Telangana,28,Female,,Active"""

In [4]:
usage_data = """usage_id,customer_id,usage_month,data_used_gb,call_minutes,sms_count
1001,101,2026-01,45,900,120
1002,102,2026-01,30,600,80
1003,103,2026-01,12,250,40
1004,104,2026-01,55,1100,150
1005,105,2026-01,75,1500,200
1006,106,2026-01,28,500,60
1007,107,2026-01,10,200,20
1008,108,2026-01,80,1600,250
1009,109,2026-01,48,950,100
1010,110,2026-01,32,700,90
1011,120,2026-01,60,1300,140
1012,101,2026-02,50,1000,130
1013,102,2026-02,34,650,85
1014,104,2026-02,58,1200,160
1015,105,2026-02,,1450,210"""

In [5]:
plans_data = [
    {"plan_id": "P101", "plan_name": "Smart Basic", "monthly_fee": 499, "data_limit_gb": 50, "features": {"unlimited_calls": True, "ott_included": False, "roaming": "National"}},
    {"plan_id": "P102", "plan_name": "Smart Plus", "monthly_fee": 799, "data_limit_gb": 75, "features": {"unlimited_calls": True, "ott_included": True, "roaming": "National"}},
    {"plan_id": "P103", "plan_name": "Budget Saver", "monthly_fee": 299, "data_limit_gb": 25, "features": {"unlimited_calls": False, "ott_included": False, "roaming": None}},
    {"plan_id": "P104", "plan_name": "Premium Max", "monthly_fee": 1199, "data_limit_gb": 100, "features": {"unlimited_calls": True, "ott_included": True, "roaming": "International"}}
]

In [6]:
payments_data = """payment_id,customer_id,bill_month,amount_paid,payment_mode,payment_status
5001,101,2026-01,499,UPI,Success
5002,102,2026-01,799,Card,Success
5003,103,2026-01,299,Cash,Failed
5004,104,2026-01,499,UPI,Success
5005,105,2026-01,1199,Card,Success
5006,106,2026-01,799,UPI,Success
5007,107,2026-01,299,Cash,Pending
5008,108,2026-01,1199,Card,Success
5009,109,2026-01,499,UPI,Success
5010,110,2026-01,799,UPI,Success
5011,112,2026-01,,UPI,Success
5012,101,2026-02,499,Card,Success
5013,102,2026-02,799,UPI,Success
5014,104,2026-02,499,UPI,Success
5015,105,2026-02,1199,,Pending"""

In [7]:
with open("customers.csv", "w") as f: f.write(customers_data)
with open("usage.csv", "w") as f: f.write(usage_data)
with open("payments.csv", "w") as f: f.write(payments_data)
with open("plans.json", "w") as f: json.dump(plans_data, f, indent=4)

In [8]:
df_customers_raw = spark.read.csv("customers.csv", header=True, inferSchema=True)
df_usage_raw = spark.read.csv("usage.csv", header=True, inferSchema=True)
df_payments_raw = spark.read.csv("payments.csv", header=True, inferSchema=True)
df_plans_raw = spark.read.option("multiLine", "true").json("plans.json")

In [9]:
print("Customers Raw Schema:")
df_customers_raw.printSchema()
print("Usage Raw Schema:")
df_usage_raw.printSchema()
print("Payments Raw Schema:")
df_payments_raw.printSchema()
print("Plans Raw Schema:")
df_plans_raw.printSchema()

Customers Raw Schema:
root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- plan_id: string (nullable = true)
 |-- status: string (nullable = true)

Usage Raw Schema:
root
 |-- usage_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- usage_month: timestamp (nullable = true)
 |-- data_used_gb: integer (nullable = true)
 |-- call_minutes: integer (nullable = true)
 |-- sms_count: integer (nullable = true)

Payments Raw Schema:
root
 |-- payment_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- bill_month: timestamp (nullable = true)
 |-- amount_paid: integer (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- payment_status: string (nullable = true)

Plans Raw Schema:
root
 |-- data_limit_gb: long (nullable = true)
 |-- features

In [10]:
print(f"Customers Count: {df_customers_raw.count()}")
print(f"Usage Count: {df_usage_raw.count()}")
print(f"Payments Count: {df_payments_raw.count()}")
print(f"Plans Count: {df_plans_raw.count()}")

Customers Count: 12
Usage Count: 15
Payments Count: 15
Plans Count: 4


In [11]:
df_customers_raw.write.mode("overwrite").parquet("bronze/customers")
df_usage_raw.write.mode("overwrite").parquet("bronze/usage")
df_payments_raw.write.mode("overwrite").parquet("bronze/payments")
df_plans_raw.write.mode("overwrite").parquet("bronze/plans")

In [12]:
df_cust = spark.read.parquet("bronze/customers")
df_usg = spark.read.parquet("bronze/usage")
df_pay = spark.read.parquet("bronze/payments")

In [13]:
print("Customers with missing plan_id:")
df_cust.filter(col("plan_id").isNull()).show()

Customers with missing plan_id:
+-----------+-------------+---------+---------+---+------+-------+------+
|customer_id|customer_name|     city|    state|age|gender|plan_id|status|
+-----------+-------------+---------+---------+---+------+-------+------+
|        112|  Ayesha Khan|Hyderabad|Telangana| 28|Female|   NULL|Active|
+-----------+-------------+---------+---------+---+------+-------+------+



In [14]:
print("Usage records with missing data_used_gb:")
df_usg.filter(col("data_used_gb").isNull()).show()

Usage records with missing data_used_gb:
+--------+-----------+-------------------+------------+------------+---------+
|usage_id|customer_id|        usage_month|data_used_gb|call_minutes|sms_count|
+--------+-----------+-------------------+------------+------------+---------+
|    1015|        105|2026-02-01 00:00:00|        NULL|        1450|      210|
+--------+-----------+-------------------+------------+------------+---------+



In [15]:
print("Payments with missing amount_paid:")
df_pay.filter(col("amount_paid").isNull()).show()

Payments with missing amount_paid:
+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|      5011|        112|2026-01-01 00:00:00|       NULL|         UPI|       Success|
+----------+-----------+-------------------+-----------+------------+--------------+



In [16]:
print("Payments with missing payment_mode:")
df_pay.filter(col("payment_mode").isNull()).show()

Payments with missing payment_mode:
+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|      5015|        105|2026-02-01 00:00:00|       1199|        NULL|       Pending|
+----------+-----------+-------------------+-----------+------------+--------------+



In [17]:
df_cust_clean = df_cust.withColumn(
    "data_quality_status",
    when(col("plan_id").isNull(), "Passed with Defaults").otherwise("Passed")
).na.fill({"plan_id": "UNKNOWN"})

In [18]:
df_usg_clean = df_usg.withColumn(
    "data_quality_status",
    when(col("data_used_gb").isNull(), "Passed with Defaults").otherwise("Passed")
).na.fill({"data_used_gb": 0})

In [19]:
df_pay_clean = df_pay.withColumn(
    "data_quality_status",
    when(col("amount_paid").isNull() | col("payment_mode").isNull(), "Passed with Defaults").otherwise("Passed")
).na.fill({"amount_paid": 0, "payment_mode": "Not Provided"})

In [20]:
df_cust_clean.write.mode("overwrite").parquet("silver/customers")
df_usg_clean.write.mode("overwrite").parquet("silver/usage")
df_pay_clean.write.mode("overwrite").parquet("silver/payments")

In [21]:
df_plans = spark.read.parquet("bronze/plans")

In [22]:
df_plans_flattened = df_plans.select(
    col("plan_id"),
    col("plan_name"),
    col("monthly_fee"),
    col("data_limit_gb"),
    col("features.unlimited_calls").alias("unlimited_calls"),
    col("features.ott_included").alias("ott_included"),
    coalesce(col("features.roaming"), lit("Not Available")).alias("roaming")
)

In [23]:
df_plans_flattened.write.mode("overwrite").parquet("silver/plans")
df_plans_flattened.show()

+-------+------------+-----------+-------------+---------------+------------+-------------+
|plan_id|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|
+-------+------------+-----------+-------------+---------------+------------+-------------+
|   P101| Smart Basic|        499|           50|           true|       false|     National|
|   P102|  Smart Plus|        799|           75|           true|        true|     National|
|   P103|Budget Saver|        299|           25|          false|       false|Not Available|
|   P104| Premium Max|       1199|          100|           true|        true|International|
+-------+------------+-----------+-------------+---------------+------------+-------------+



In [24]:
df_c = spark.read.parquet("silver/customers")
df_u = spark.read.parquet("silver/usage")
df_p = spark.read.parquet("silver/payments")
df_pl = spark.read.parquet("silver/plans")

In [25]:
df_cust_plan = df_c.join(df_pl, on="plan_id", how="left")

In [26]:
df_usage_pay = df_u.join(df_p, (df_u.customer_id == df_p.customer_id) & (df_u.usage_month == df_p.bill_month), how="full") \
    .select(
        coalesce(df_u.customer_id, df_p.customer_id).alias("customer_id"),
        coalesce(df_u.usage_month, df_p.bill_month).alias("usage_month"),
        df_u.usage_id, df_u.data_used_gb, df_u.call_minutes, df_u.sms_count,
        df_p.payment_id, df_p.amount_paid, df_p.payment_mode, df_p.payment_status
    )

df_master = df_cust_plan.join(df_usage_pay, on="customer_id", how="full")
df_master.cache()

DataFrame[customer_id: int, plan_id: string, customer_name: string, city: string, state: string, age: int, gender: string, status: string, data_quality_status: string, plan_name: string, monthly_fee: bigint, data_limit_gb: bigint, unlimited_calls: boolean, ott_included: boolean, roaming: string, usage_month: timestamp, usage_id: int, data_used_gb: int, call_minutes: int, sms_count: int, payment_id: int, amount_paid: int, payment_mode: string, payment_status: string]

In [27]:
print("Customers with invalid Plan IDs:")
df_c.join(df_pl, on="plan_id", how="left_anti").filter(col("plan_id") != "UNKNOWN").show()

Customers with invalid Plan IDs:
+-------+-----------+-------------+------+-----------+---+------+------+-------------------+
|plan_id|customer_id|customer_name|  city|      state|age|gender|status|data_quality_status|
+-------+-----------+-------------+------+-----------+---+------+------+-------------------+
|   P105|        111|   Ravi Kumar|Mumbai|Maharashtra| 45|  Male|Active|             Passed|
+-------+-----------+-------------+------+-----------+---+------+------+-------------------+



In [28]:
print("Usage records without matching customer:")
df_u.join(df_c, on="customer_id", how="left_anti").show()

Usage records without matching customer:
+-----------+--------+-------------------+------------+------------+---------+-------------------+
|customer_id|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|
+-----------+--------+-------------------+------------+------------+---------+-------------------+
|        120|    1011|2026-01-01 00:00:00|          60|        1300|      140|             Passed|
+-----------+--------+-------------------+------------+------------+---------+-------------------+



In [29]:
print("Payments without matching customer:")
df_p.join(df_c, on="customer_id", how="left_anti").show()

Payments without matching customer:
+-----------+----------+----------+-----------+------------+--------------+-------------------+
|customer_id|payment_id|bill_month|amount_paid|payment_mode|payment_status|data_quality_status|
+-----------+----------+----------+-----------+------------+--------------+-------------------+
+-----------+----------+----------+-----------+------------+--------------+-------------------+



In [30]:
df_transformed = df_master.withColumn(
    "usage_category",
    when(col("data_used_gb") >= 70, "Heavy User")
    .when(col("data_used_gb") >= 30, "Medium User")
    .otherwise("Low User")
).withColumn(
    # 32. Create payment_category
    "payment_category",
    when(col("amount_paid") >= 1000, "High Payment")
    .when(col("amount_paid") >= 500, "Medium Payment")
    .otherwise("Low Payment")
).withColumn(
    # 33. Create churn_risk profile logic
    "churn_risk",
    when((col("status") == "Inactive") | (col("payment_status") == "Failed") | (col("payment_status") == "Pending"), "High Risk")
    .when(col("data_used_gb") < 15, "Medium Risk")
    .otherwise("Low Risk")
).withColumn(
    # 34. Create over_usage_gb
    "over_usage_gb",
    when(col("plan_id") == "UNKNOWN", 0)
    .otherwise(coalesce(col("data_used_gb"), lit(0)) - coalesce(col("data_limit_gb"), lit(0)))
).withColumn(
    # 35. Create over_usage_flag
    "over_usage_flag",
    when(col("over_usage_gb") > 0, "Yes").otherwise("No")
)

In [34]:
try:
    df_t = spark.read.parquet("silver/transformed_master")
except Exception:
    print("Silver directory not found on disk. Using in-memory transformed DataFrame.")
    df_t = df_transformed
df_unique_cust = df_t.dropDuplicates(["customer_id"])


Silver directory not found on disk. Using in-memory transformed DataFrame.


In [35]:
import os

# Create local directories if they don't exist yet
os.makedirs("bronze", exist_ok=True)
os.makedirs("silver", exist_ok=True)
os.makedirs("gold", exist_ok=True)

# Save the base clean dataframes so Parts 8 & 10 can find them safely
df_cust_clean.write.mode("overwrite").parquet("silver/customers")
df_usg_clean.write.mode("overwrite").parquet("silver/usage")
df_pay_clean.write.mode("overwrite").parquet("silver/payments")
df_plans_flattened.write.mode("overwrite").parquet("silver/plans")

In [36]:
df_unique_cust = df_t.dropDuplicates(["customer_id"])

In [37]:
df_unique_cust.groupBy("city").count().alias("Customers_By_City").show(3)
df_unique_cust.groupBy("state").count().alias("Customers_By_State").show(3)
df_unique_cust.groupBy("plan_name").count().alias("Customers_By_Plan").show(3)
df_t.groupBy("usage_category").count().alias("Records_By_Usage_Category").show(3)
df_t.groupBy("churn_risk").count().alias("Records_By_Churn_Risk").show(3)

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|     Pune|    1|
|    Delhi|    2|
+---------+-----+
only showing top 3 rows
+-----------+-----+
|      state|count|
+-----------+-----+
|  Telangana|    3|
|Maharashtra|    3|
|  Karnataka|    2|
+-----------+-----+
only showing top 3 rows
+------------+-----+
|   plan_name|count|
+------------+-----+
| Premium Max|    2|
|  Smart Plus|    3|
|Budget Saver|    2|
+------------+-----+
only showing top 3 rows
+--------------+-----+
|usage_category|count|
+--------------+-----+
|   Medium User|    9|
|      Low User|    6|
|    Heavy User|    2|
+--------------+-----+

+----------+-----+
|churn_risk|count|
+----------+-----+
| High Risk|    3|
|  Low Risk|   14|
+----------+-----+



In [38]:
df_t.groupBy("plan_name").agg(
    sum("data_used_gb").alias("total_data_usage"),
    avg("data_used_gb").alias("average_data_usage")
).show()

+------------+----------------+------------------+
|   plan_name|total_data_usage|average_data_usage|
+------------+----------------+------------------+
|Budget Saver|              22|              11.0|
|  Smart Plus|             124|              31.0|
| Smart Basic|             256|              51.2|
| Premium Max|             155|51.666666666666664|
|        NULL|              60|              60.0|
+------------+----------------+------------------+



In [39]:
df_t.groupBy("city").agg(sum("call_minutes").alias("total_call_minutes")).show(3)

+---------+------------------+
|     city|total_call_minutes|
+---------+------------------+
|Bangalore|              2200|
|    Delhi|              3650|
|Hyderabad|              2100|
+---------+------------------+
only showing top 3 rows


In [40]:
df_t.groupBy("state").agg(sum("sms_count").alias("total_sms_count")).show(3)

+---------+---------------+
|    state|total_sms_count|
+---------+---------------+
|Telangana|            270|
|Karnataka|            265|
|    Delhi|            500|
+---------+---------------+
only showing top 3 rows


In [41]:
successful_rev = df_t.filter(col("payment_status") == "Success").agg(sum("amount_paid")).collect()[0][0]
print(f"Total Successful Revenue: INR {successful_rev}")

Total Successful Revenue: INR 8089


In [42]:
df_rev_city = df_t.filter(col("payment_status") == "Success").groupBy("city").agg(sum("amount_paid").alias("revenue"))
df_rev_plan = df_t.filter(col("payment_status") == "Success").groupBy("plan_name").agg(sum("amount_paid").alias("revenue"))
df_rev_mode = df_t.filter(col("payment_status") == "Success").groupBy("payment_mode").agg(sum("amount_paid").alias("revenue"))



In [43]:
df_rev_city.show(3)
df_rev_plan.show(3)

+---------+-------+
|     city|revenue|
+---------+-------+
|Bangalore|   2097|
|    Delhi|   1998|
|Hyderabad|    998|
+---------+-------+
only showing top 3 rows
+-----------+-------+
|  plan_name|revenue|
+-----------+-------+
| Smart Plus|   3196|
|Smart Basic|   2495|
|Premium Max|   2398|
+-----------+-------+
only showing top 3 rows


In [44]:
print("Plan with highest revenue:")
df_rev_plan.orderBy(col("revenue").desc()).show(1)

Plan with highest revenue:
+----------+-------+
| plan_name|revenue|
+----------+-------+
|Smart Plus|   3196|
+----------+-------+
only showing top 1 row


In [45]:
print("City with highest revenue:")
df_rev_city.orderBy(col("revenue").desc()).show(1)

City with highest revenue:
+---------+-------+
|     city|revenue|
+---------+-------+
|Bangalore|   2097|
+---------+-------+
only showing top 1 row


In [73]:
win_data = Window.partitionBy("usage_month").orderBy(col("data_used_gb").desc())
win_pay = Window.partitionBy("usage_month").orderBy(col("amount_paid").desc())
win_city = Window.partitionBy("city").orderBy(col("data_used_gb").desc())
win_plan = Window.partitionBy("plan_name").orderBy(col("data_used_gb").desc())
win_timeline = Window.partitionBy("customer_id").orderBy("usage_month")

In [74]:
df_windowed = df_t.withColumn("data_rank", rank().over(win_data))
df_windowed = df_windowed.withColumn("pay_rank", rank().over(win_pay))

In [75]:
print("Top 3 data users per month:")
df_windowed.filter(col("data_rank") <= 3) \
           .select("usage_month", "customer_name", "data_used_gb", "data_rank") \
           .show(5)

Top 3 data users per month:
+-------------------+-------------+------------+---------+
|        usage_month|customer_name|data_used_gb|data_rank|
+-------------------+-------------+------------+---------+
|               NULL|   Ravi Kumar|        NULL|        1|
|2026-01-01 00:00:00|   Meera Nair|          80|        1|
|2026-01-01 00:00:00|   Farhan Ali|          75|        2|
|2026-01-01 00:00:00|         NULL|          60|        3|
|2026-02-01 00:00:00|  Sneha Patel|          58|        1|
+-------------------+-------------+------------+---------+
only showing top 5 rows


In [76]:
print("Top 3 revenue customers per month:")
df_windowed.filter(col("pay_rank") <= 3) \
           .select("usage_month", "customer_name", "amount_paid", "pay_rank") \
           .show(5)

Top 3 revenue customers per month:
+-------------------+-------------+-----------+--------+
|        usage_month|customer_name|amount_paid|pay_rank|
+-------------------+-------------+-----------+--------+
|               NULL|   Ravi Kumar|       NULL|       1|
|2026-01-01 00:00:00|   Farhan Ali|       1199|       1|
|2026-01-01 00:00:00|   Meera Nair|       1199|       1|
|2026-01-01 00:00:00|  Nisha Reddy|        799|       3|
|2026-01-01 00:00:00|   Neha Singh|        799|       3|
+-------------------+-------------+-----------+--------+
only showing top 5 rows


In [50]:
print("Top 3 revenue customers per month:")
df_windowed.filter(col("pay_rank") <= 3).select("usage_month", "customer_name", "amount_paid").show(5)

Top 3 revenue customers per month:
+-------------------+-------------+-----------+
|        usage_month|customer_name|amount_paid|
+-------------------+-------------+-----------+
|               NULL|   Ravi Kumar|       NULL|
|2026-01-01 00:00:00|   Farhan Ali|       1199|
|2026-01-01 00:00:00|   Meera Nair|       1199|
|2026-01-01 00:00:00|  Nisha Reddy|        799|
|2026-01-01 00:00:00|   Neha Singh|        799|
+-------------------+-------------+-----------+
only showing top 5 rows


In [51]:
print("Top customer by city:")
df_t.withColumn("city_rank", rank().over(win_city)).filter(col("city_rank") == 1).select("city", "customer_name", "data_used_gb").show(3)

Top customer by city:
+---------+-------------+------------+
|     city|customer_name|data_used_gb|
+---------+-------------+------------+
|     NULL|         NULL|          60|
|Bangalore|    Kiran Rao|          48|
|  Chennai|  Sneha Patel|          58|
+---------+-------------+------------+
only showing top 3 rows


In [52]:
print("Top customer by plan:")
df_t.withColumn("plan_rank", rank().over(win_plan)).filter(col("plan_rank") == 1).select("plan_name", "customer_name", "data_used_gb").show(3)

Top customer by plan:
+------------+-------------+------------+
|   plan_name|customer_name|data_used_gb|
+------------+-------------+------------+
|        NULL|         NULL|          60|
|Budget Saver|   Amit Kumar|          12|
| Premium Max|   Meera Nair|          80|
+------------+-------------+------------+
only showing top 3 rows


In [53]:
win_run_total = Window.orderBy("usage_month").rowsBetween(Window.unboundedPreceding, Window.currentRow)
df_t.filter(col("payment_status") == "Success") \
    .groupBy("usage_month").agg(sum("amount_paid").alias("monthly_rev")) \
    .withColumn("running_total_revenue", sum("monthly_rev").over(win_run_total)).show()

+-------------------+-----------+---------------------+
|        usage_month|monthly_rev|running_total_revenue|
+-------------------+-----------+---------------------+
|2026-01-01 00:00:00|       6292|                 6292|
|2026-02-01 00:00:00|       1797|                 8089|
+-------------------+-----------+---------------------+



In [54]:
df_time_series = df_t.filter(col("customer_id").isNotNull() & col("usage_month").isNotNull()) \
    .withColumn("prev_month_usage", lag("data_used_gb", 1).over(win_timeline)) \
    .withColumn("next_month_usage", lead("data_used_gb", 1).over(win_timeline))

In [55]:
print("Customers whose usage increased month over month:")
df_time_series.filter(col("data_used_gb") > col("prev_month_usage")) \
              .select("customer_id", "customer_name", "usage_month", "prev_month_usage", "data_used_gb").show()

Customers whose usage increased month over month:
+-----------+-------------+-------------------+----------------+------------+
|customer_id|customer_name|        usage_month|prev_month_usage|data_used_gb|
+-----------+-------------+-------------------+----------------+------------+
|        101| Rahul Sharma|2026-02-01 00:00:00|              45|          50|
|        102|  Priya Reddy|2026-02-01 00:00:00|              30|          34|
|        104|  Sneha Patel|2026-02-01 00:00:00|              55|          58|
+-----------+-------------+-------------------+----------------+------------+



In [56]:
df_t.write.mode("overwrite").partitionBy("usage_month").parquet("gold/customer_usage_summary")

In [57]:
inc_usage_data = """usage_id,customer_id,usage_month,data_used_gb,call_minutes,sms_count
1016,101,2026-03,62,1100,140
1017,102,2026-03,42,720,95
1018,104,2026-03,65,1300,180"""
with open("usage_inc_2026_03.csv", "w") as f: f.write(inc_usage_data)

In [58]:
df_inc_usg_raw = spark.read.csv("usage_inc_2026_03.csv", header=True, inferSchema=True)

In [59]:
df_inc_usg_clean = df_inc_usg_raw.withColumn("data_quality_status", lit("Passed")).na.fill({"data_used_gb": 0})
df_inc_usg_clean.write.mode("append").parquet("silver/usage")

In [60]:
df_u_updated = spark.read.parquet("silver/usage")
df_pay_updated = spark.read.parquet("silver/payments")

In [61]:
df_usage_pay_updated = df_u_updated.join(df_pay_updated, (df_u_updated.customer_id == df_pay_updated.customer_id) & (df_u_updated.usage_month == df_pay_updated.bill_month), how="full") \
    .select(
        coalesce(df_u_updated.customer_id, df_pay_updated.customer_id).alias("customer_id"),
        coalesce(df_u_updated.usage_month, df_pay_updated.bill_month).alias("usage_month"),
        df_u_updated.data_used_gb, df_u_updated.call_minutes, df_u_updated.sms_count,
        df_pay_updated.amount_paid, df_pay_updated.payment_status, df_pay_updated.payment_mode
    )

In [62]:
df_gold_recalculated = df_c.join(df_pl, on="plan_id", how="left") \
    .join(df_usage_pay_updated, on="customer_id", how="full") \
    .withColumn("usage_category", when(col("data_used_gb") >= 70, "Heavy User").when(col("data_used_gb") >= 30, "Medium User").otherwise("Low User")) \
    .withColumn("payment_category", when(col("amount_paid") >= 1000, "High Payment").when(col("amount_paid") >= 500, "Medium Payment").otherwise("Low Payment")) \
    .withColumn("churn_risk", when((col("status") == "Inactive") | (col("payment_status") == "Failed") | (col("payment_status") == "Pending"), "High Risk").when(col("data_used_gb") < 15, "Medium Risk").otherwise("Low Risk")) \
    .withColumn("over_usage_gb", when(col("plan_id") == "UNKNOWN", 0).otherwise(coalesce(col("data_used_gb"), lit(0)) - coalesce(col("data_limit_gb"), lit(0)))) \
    .withColumn("over_usage_flag", when(col("over_usage_gb") > 0, "Yes").otherwise("No"))

In [63]:
df_gold_recalculated.write.mode("overwrite").partitionBy("usage_month").parquet("gold/customer_usage_summary")

In [64]:
print(f"Original Row Count in Master Transform: {df_t.count()}")
print(f"Updated Post-Incremental Gold Row Count: {df_gold_recalculated.count()}")

Original Row Count in Master Transform: 20
Updated Post-Incremental Gold Row Count: 20


In [65]:
df_gold = spark.read.parquet("gold/customer_usage_summary")

In [66]:
customer_usage_summary_report = df_gold.select(
    "customer_id", "customer_name", "city", "plan_name", "usage_month",
    "data_used_gb", "data_limit_gb", "over_usage_flag", "amount_paid", "payment_status", "churn_risk"
)
print("--- 79. Customer Usage Summary Report ---")
customer_usage_summary_report.show(5)

--- 79. Customer Usage Summary Report ---
+-----------+-------------+---------+------------+-------------------+------------+-------------+---------------+-----------+--------------+----------+
|customer_id|customer_name|     city|   plan_name|        usage_month|data_used_gb|data_limit_gb|over_usage_flag|amount_paid|payment_status|churn_risk|
+-----------+-------------+---------+------------+-------------------+------------+-------------+---------------+-----------+--------------+----------+
|        102|  Priya Reddy|Bangalore|  Smart Plus|2026-01-01 00:00:00|          30|           75|             No|        799|       Success|  Low Risk|
|        103|   Amit Kumar|   Mumbai|Budget Saver|2026-01-01 00:00:00|          12|           25|             No|        299|        Failed| High Risk|
|        104|  Sneha Patel|  Chennai| Smart Basic|2026-01-01 00:00:00|          55|           50|            Yes|        499|       Success|  Low Risk|
|        105|   Farhan Ali|    Delhi| Premium 

In [68]:
plan_performance_report = df_gold.groupBy("plan_name").agg(
    count_distinct("customer_id").alias("total_customers"),
    sum("data_used_gb").alias("total_data_usage"),
    avg("data_used_gb").alias("average_data_usage"),
    sum(when(col("payment_status") == "Success", col("amount_paid")).otherwise(0)).alias("total_revenue")
)
plan_performance_report.show()

+------------+---------------+----------------+------------------+-------------+
|   plan_name|total_customers|total_data_usage|average_data_usage|total_revenue|
+------------+---------------+----------------+------------------+-------------+
| Premium Max|              2|             155|51.666666666666664|         2398|
|Budget Saver|              2|              22|              11.0|            0|
|  Smart Plus|              3|             166|              33.2|         3196|
| Smart Basic|              3|             383|54.714285714285715|         2495|
|        NULL|              3|              60|              60.0|            0|
+------------+---------------+----------------+------------------+-------------+



In [69]:
city_revenue_report = df_gold.groupBy("city").agg(
    count_distinct("customer_id").alias("total_customers"),
    sum(when(col("payment_status") == "Success", col("amount_paid")).otherwise(0)).alias("total_revenue"),
    avg("amount_paid").alias("average_payment")
)
city_revenue_report.show()

+---------+---------------+-------------+------------------+
|     city|total_customers|total_revenue|   average_payment|
+---------+---------------+-------------+------------------+
|     Pune|              1|          799|             799.0|
|Bangalore|              2|         2097|             699.0|
|    Delhi|              2|         1998|1065.6666666666667|
|Hyderabad|              3|          998|            324.25|
|     NULL|              1|            0|              NULL|
|   Mumbai|              2|            0|             299.0|
|    Kochi|              1|         1199|            1199.0|
|  Chennai|              1|          998|             499.0|
+---------+---------------+-------------+------------------+



In [70]:
churn_risk_report = df_gold.select(
    "customer_id", "customer_name", "city", "plan_name", "payment_status", "status", "churn_risk"
)
churn_risk_report.show(5)

+-----------+-------------+---------+------------+--------------+--------+----------+
|customer_id|customer_name|     city|   plan_name|payment_status|  status|churn_risk|
+-----------+-------------+---------+------------+--------------+--------+----------+
|        102|  Priya Reddy|Bangalore|  Smart Plus|       Success|  Active|  Low Risk|
|        103|   Amit Kumar|   Mumbai|Budget Saver|        Failed|Inactive| High Risk|
|        104|  Sneha Patel|  Chennai| Smart Basic|       Success|  Active|  Low Risk|
|        105|   Farhan Ali|    Delhi| Premium Max|       Success|  Active|  Low Risk|
|        106|   Neha Singh|     Pune|  Smart Plus|       Success|  Active|  Low Risk|
+-----------+-------------+---------+------------+--------------+--------+----------+
only showing top 5 rows


In [71]:
over_usage_report = df_gold.filter(col("over_usage_gb") > 0).select(
    "customer_id", "customer_name", "plan_name", "data_used_gb", "data_limit_gb", "over_usage_gb"
)
over_usage_report.show(5)

+-----------+-------------+-----------+------------+-------------+-------------+
|customer_id|customer_name|  plan_name|data_used_gb|data_limit_gb|over_usage_gb|
+-----------+-------------+-----------+------------+-------------+-------------+
|        104|  Sneha Patel|Smart Basic|          55|           50|            5|
|        120|         NULL|       NULL|          60|         NULL|           60|
|        104|  Sneha Patel|Smart Basic|          58|           50|            8|
|        101| Rahul Sharma|Smart Basic|          62|           50|           12|
|        104|  Sneha Patel|Smart Basic|          65|           50|           15|
+-----------+-------------+-----------+------------+-------------+-------------+



In [ ]:
print("Capstone Data Pipeline Executed Successfully!")